# SEGUIMIENTO DE AUTORIZACIONES

## Ejercicio 2025-2026

In [1]:
# Control de actualización de los procesos 🎛️
from mstrio.project_objects import Report
from utils.login import conn

report_id = '96B3AA41488F3ABAE3C37C97DB658B23'
tabla_control = Report(id=report_id, connection=conn, progress_bar=False).to_dataframe()

# IDs que te interesan
ids_interes = ['6','37','14', '17']
tabla_control = tabla_control[tabla_control['Nombre de Proceso@ID'].isin(ids_interes)]
tabla_control = tabla_control[['Nombre de Proceso@DESC', 'Ultima Fecha Actualización de Proceso']].rename(columns={
    'Nombre de Proceso@DESC': 'Proceso',
    'Ultima Fecha Actualización de Proceso': 'Última Actualización'
})

tabla_control

Connection to Strategy One Intelligence Server has been established.
Project selected in Connection object:
Project object named: 'Salud' with ID: 'DAE6DF9811D67BD9500010A51D1D2ADA'
Report object named: 'Tabla plana Actualización de Procesos' with ID: '96B3AA41488F3ABAE3C37C97DB658B23'


,Proceso,Última Actualización
3,Conectividad,6/19/2026 6:05:04 AM
8,Consumo de Medicamentos,6/19/2026 5:39:27 AM
11,Internaciones Diario,6/19/2026 5:25:00 AM
21,Autorizaciones Ordenes Stock Diario,6/18/2026 9:14:55 PM


In [2]:
# Ejecutar scripts externos ⚠️
!python sync_dw_to_sqlite_u6m.py
!python utils/backup.py

Connection to Strategy One Intelligence Server has been established.
Project selected in Connection object:
Project object named: 'Salud' with ID: 'DAE6DF9811D67BD9500010A51D1D2ADA'
🗓️ Fecha de hoy: 2026-06-19
🧮 Cantidad de registros eliminados en ambulatorio: 289763
🧮 Cantidad de registros eliminados en farmvac: 14863
🗑️ Registros eliminados desde 2025-12-19 en la tabla ambulatorio y farmvac.
Report object named: '01 - Autorizaciones AMB - u6m' with ID: 'A56FCF021448592FE462F582E9446136'
⚙️ Cantidad de registros en Autorizaciones AMBULATORIO F57 en los U6M: 221118
Report object named: '02 - Autorizaciones AMB F4 - u6m' with ID: '2C938EDEFE42E6C843E10FBACFA21A8B'
⚙️ Cantidad de registros en Autorizaciones AMBULATORIO F4 en los U6M: 70828
Report object named: '03 - Autorizaciones MFARM - u6m' with ID: 'E9BB407DF04FA120D71037BB096A9E69'
⚙️ Cantidad de registros en Autorizaciones FARMACIA en los U6M: 12591
Report object named: '04 - Autorizaciones MVAC -u6m' with ID: 'F214594EBE4A4BE491C1

🗃️ Backup realizado: backups\sda_2025_2026_backup_2026-06-19.sqlite


Librerias Python

In [3]:
# Importando librerías necesarias 📚
import sqlite3
import pandas as pd
from datetime import datetime, timedelta
import holidays
import numpy as np
import os

AGREGADO PARCIAL DE RESTO

In [4]:
# Cargando reporte RESTO
from mstrio.project_objects import Report
from utils.login import conn

report_id = '52514B39A84BDA359C68C3A3CBC46E62'
resto = Report(id=report_id, connection=conn).to_dataframe()

Report object named: '02 - Consultas Resto' with ID: '52514B39A84BDA359C68C3A3CBC46E62'


In [5]:
# Resto (Rubro Prestación Gerencia Estrategic: F/D & Resto - Prestación DESC contiene "CONSULTA" )
r = resto.copy()
r = r.rename(columns={
    'Fecha Proceso Autorización':'Fecha',
    'Rubro Prestacion Gerencia Estrategica':'Rubro',
    'Subrubro Prestacion Gerencia Estrategica':'Subrubro',
    'Zona Direccion Comercial Asociado':'Zona',
    'Subzona Direccion Comercial Asociado':'Subzona',
    'Cantidad Prestaciones Aceptadas':'Prestaciones',
    'Tipo Orden' : 'Transaccion'
})

r['Fecha'] = pd.to_datetime(r['Fecha'])
r['Periodo'] = r['Periodo'].astype(int)
r['Rubro'] = r['Rubro'].replace({'F/D':'Consulta','Resto':'Consulta'})

# Reemplazar 'Subrubro' en partes iguales entre "Con. Guardia" y "Con. Programada"
n = len(r)
rng = np.random.default_rng(42)  # semilla para reproducibilidad
perm = rng.permutation(n)
half = n // 2

guardia_idx = r.iloc[perm[:half]].index
programada_idx = r.iloc[perm[half:]].index

r.loc[guardia_idx, 'Subrubro'] = 'Con. Guardia'
r.loc[programada_idx, 'Subrubro'] = 'Con. Programada'

Fecha de hoy

In [6]:
# Cargando fecha de hoy y periodo actual ⏳
hoy = pd.Timestamp.today().normalize() # Fecha de hoy
print(f"Fecha de hoy: {hoy}")

ayer = hoy - timedelta(days=1) # Fecha de ayer
print(f"Fecha de ayer: {ayer}")

dia_hoy = hoy.strftime('%Y-%m-%d') # Fecha de hoy en formato string 'YYYY-MM-DD'

dias_proy = hoy - timedelta(days=30) # Fecha hace 30 días (Ajustable para la cantidad de dias que se quiera proyectar)
dias_proy = pd.Timestamp(dias_proy).normalize() # Normalizar
print(f"Fecha hace 30 días: {dias_proy}")
periodo_actual = int(f"{hoy.year}{hoy.month:02d}") # Periodo actual (por ejemplo, 202508)
print(f"Periodo actual: {periodo_actual}")

Fecha de hoy: 2026-06-19 00:00:00
Fecha de ayer: 2026-06-18 00:00:00
Fecha hace 30 días: 2026-05-20 00:00:00
Periodo actual: 202606


Feriados Nacionales Argentina 2025-2026

In [7]:
# Creando calendario con los feriados de 2025 y 2026 📅
anios = [2025, 2026]
feriados_ar = holidays.AR(years=anios)
feriados = pd.to_datetime(list(feriados_ar.keys()))
excluidos = ['2025-09-23','2025-10-07','2025-10-10', '2025-12-24','2025-12-31'] #días excluidos de manera manual
feriados = feriados.append(pd.to_datetime(excluidos))
# --- para mostrar en las proyecciones ---
no_laborables = ['2025-11-21','2026-03-23']
feriado_puro = ['2025-11-24','2025-12-08','2025-12-25','2026-02-16','2026-02-17','2026-03-24', '2026-05-25', '2026-06-15','2026-06-20']
fiestas = ['2025-12-24','2025-12-31']

Conexión a la base de datos

In [8]:
# Conectando a la base de datos SQLite 💽
conn = sqlite3.connect('sda_2025_2026.sqlite')

# Leer las tablas necesarias
ambulatorio = pd.read_sql_query("SELECT * FROM ambulatorio;", conn)
farmvac = pd.read_sql_query("SELECT * FROM farmvac;", conn)
provision = pd.read_sql_query("SELECT * FROM provision;", conn)
discapacidad = pd.read_sql_query("SELECT * FROM discapacidad;", conn)
protesis = pd.read_sql_query("SELECT * FROM protesis;", conn)


# Asegurarte de que 'Fecha' sea datetime
ambulatorio['Fecha'] = pd.to_datetime(ambulatorio['Fecha'])
farmvac['Fecha'] = pd.to_datetime(farmvac['Fecha'])

# Filtrar excluyendo el día de hoy
rubros_filtrar = [
    "Consulta", "Practicas", "Optica", "Fisiokinesiologia",
    "Salud Mental", "Imagenes", "Cirugia", "Laboratorio"
]

# AGREGADO PARCIAL DE 'RESTO' HASTA LA CORRECCIÓN DE LOS RUBROS Y SUBRUBROS
amb_resto = pd.concat([ambulatorio, r]) # une ambos DataFrames

# Columnas a conservar (definidas una sola vez)
columnas_amb = ['Periodo','Fecha','Zona','Subzona','Transaccion','Rubro','Subrubro','Prestaciones']
columnas_farmvac = ['Periodo','Fecha','Zona','Subzona','Origen','Cantidad','Importe','Rubro']

# Filtrado y selección de columnas en una sola operación
ambulatorio_wtf = amb_resto.loc[
    (amb_resto['Fecha'] < hoy) & 
    (amb_resto['Rubro'].isin(rubros_filtrar)),
    columnas_amb
]

farmvac_wt = farmvac.loc[
    farmvac['Fecha'] < hoy,
    columnas_farmvac
]

### AMBULATORIO

#### Proyección diaria
- __Intervalo de tiempo__: días del corriente mes.   
- __Apertura__: Rubro  
- __Métricas__: Nivel Real - Nivel Proyectado - Nivel Esperado - Acumulado Real - Acumulado Proyectado  
<br>
_Para calcular los Niveles Proyectados se toman los últimos 28 días sin contar el actual._  

In [9]:
# Calculando proyección diaria de ambulatorio

auxiliares = pd.read_excel('auxiliar.xlsx', sheet_name=['aux_rubro_amb', 'aux_subrubro_amb', 'amb','dif_sim_amb', 'val_amb'])

# === Función para mapeo de rubros y subrubros ===
def mapear_rubros_subrubros(df, auxiliares):
    """
    Mapea rubros y subrubros de forma eficiente usando los datos auxiliares.
    """
    # Mapeo de rubros
    dict_rubros = dict(zip(
        auxiliares['aux_rubro_amb']["Rubro Prestación Gerencia Estratégica"], 
        auxiliares['aux_rubro_amb']["Rubro Presupuesto"]
    ))
    df["Rubro Presupuesto"] = df["Rubro"].map(dict_rubros)
    
    # Mapeo de subrubros
    dict_subrubros = dict(zip(
        auxiliares['aux_subrubro_amb']["Rubro Prestacion Gerencia Estrategica&Subrubro Prestacion Gerencia Estrategica"], 
        auxiliares['aux_subrubro_amb']["Subrubro Presupuesto"]
    ))
    df["Subrubro Presupuesto"] = (df["Rubro"] + df["Subrubro"]).map(dict_subrubros)
    
    # Completar NaN en subrubros
    dict_subrubro_nan = dict(zip(
        auxiliares['aux_rubro_amb']["Rubro Presupuesto"], 
        auxiliares['aux_rubro_amb']["Subrubros para NaN"]
    ))
    mask_nan = df["Subrubro Presupuesto"].isna()
    df.loc[mask_nan, "Subrubro Presupuesto"] = df.loc[mask_nan, "Rubro Presupuesto"].map(dict_subrubro_nan)
    
    return df

# === Aplicar mapeo ===
ambulatorio_wtf = mapear_rubros_subrubros(ambulatorio_wtf, auxiliares)

# === Preparar datos históricos ===
# Filtrar y agrupar en una sola operación
apd = (
    ambulatorio_wtf
    .query('Fecha >= @dias_proy')  # Más eficiente que .loc
    [['Fecha', 'Rubro Presupuesto', 'Subrubro Presupuesto', 'Prestaciones']]
    .groupby(['Fecha', 'Rubro Presupuesto', 'Subrubro Presupuesto'], as_index=False)
    .agg({'Prestaciones': 'sum'})
)

# Agregar columnas calculadas de una vez
apd = apd.assign(
    Dia_Semana=apd["Fecha"].dt.day_name(),
    Es_Feriado=apd["Fecha"].isin(feriados)
)

# === Calcular promedios históricos ===
prom_semana = (
    apd
    .query('~Es_Feriado')  # Excluir feriados
    .groupby(["Rubro Presupuesto", "Subrubro Presupuesto", "Dia_Semana"], as_index=False)
    .agg({'Prestaciones': 'mean'})
    .rename(columns={"Prestaciones": "Promedio_Semanal"})
    .round()
)

# === Crear calendario del mes ===
ultima_fecha = apd["Fecha"].max()
inicio_mes = ultima_fecha.replace(day=1)
fin_mes = ultima_fecha + pd.offsets.MonthEnd(0)

df_cal = pd.DataFrame({
    "Fecha": pd.date_range(inicio_mes, fin_mes)
})
df_cal = df_cal.assign(
    Dia_Semana=df_cal["Fecha"].dt.day_name(),
    Es_Feriado=df_cal["Fecha"].isin(feriados)
)

# === Generar proyección final (optimizada) ===
# Obtener combinaciones únicas de rubro/subrubro
combinaciones = apd[["Rubro Presupuesto", "Subrubro Presupuesto"]].drop_duplicates()

# Crear producto cartesiano eficiente
proyamb_base = (
    df_cal
    .assign(key=1)
    .merge(combinaciones.assign(key=1), on='key')
    .drop('key', axis=1)
)

# Unir prestaciones históricas
proyamb_base = proyamb_base.merge(
    apd[["Fecha", "Rubro Presupuesto", "Subrubro Presupuesto", "Prestaciones"]],
    on=["Fecha", "Rubro Presupuesto", "Subrubro Presupuesto"],
    how="left"
)

# Unir promedios semanales
proyamb_base = proyamb_base.merge(
    prom_semana,
    on=["Rubro Presupuesto", "Subrubro Presupuesto", "Dia_Semana"],
    how="left"
)

# Asignar nivel proyectado
proyamb_base["Nivel Proyectado"] = proyamb_base["Promedio_Semanal"]

# Ajuste por feriados de las cantidades proyectadas
mascara_feriado_puro = proyamb_base['Fecha'].isin(feriado_puro)
proyamb_base["Nivel Proyectado"] = np.where(mascara_feriado_puro, (proyamb_base["Nivel Proyectado"]*0.10), proyamb_base["Nivel Proyectado"])
mascara_nl = proyamb_base['Fecha'].isin(no_laborables)
proyamb_base["Nivel Proyectado"] = np.where(mascara_nl, (proyamb_base["Nivel Proyectado"]*0.50), proyamb_base["Nivel Proyectado"])
mascara_fiestas = proyamb_base['Fecha'].isin(fiestas)
proyamb_base["Nivel Proyectado"] = np.where(mascara_fiestas, (proyamb_base["Nivel Proyectado"]*0.25), proyamb_base["Nivel Proyectado"])

# === Agregación final ===
proyamb_dia = (
    proyamb_base
    .groupby(['Fecha', 'Rubro Presupuesto', 'Subrubro Presupuesto'], as_index=False)
    .agg({'Prestaciones': 'sum', 'Nivel Proyectado': 'sum'})
    .sort_values(['Rubro Presupuesto', 'Fecha'])
    .reset_index(drop=True)
    .fillna(0)
    .astype({'Nivel Proyectado': int, 'Prestaciones': int})
)

# === Niveles Esperados ===
ne_amb = (
    auxiliares['amb']
    .query(f'Periodo == {periodo_actual}')
    .groupby('Subrubro', as_index=False)
    .agg({'Nivel Esperado': 'sum'})
    .astype({'Nivel Esperado': int})
)

proyamb_dia = (
    proyamb_dia
    .merge(ne_amb, left_on='Subrubro Presupuesto', right_on='Subrubro', how='left')
    .drop(columns=['Subrubro'])
    .rename(columns={'Subrubro Presupuesto': 'Subrubro', 'Prestaciones': 'Nivel Real'})
)

proyamb_dia['Acumulado Real'] = proyamb_dia.groupby('Subrubro')['Nivel Real'].cumsum()

# Calcular acumulado proyectado
proyamb_dia["Acumulado Proyectado"] = 0  # creamos la columna

for rubro, grupo in proyamb_dia.groupby("Subrubro"):
    # último acumulado real existente
    acumulado_inicial = grupo["Acumulado Real"].replace(0, pd.NA).ffill().iloc[-1]

    # acumulamos los proyectados
    acumulado = acumulado_inicial
    acumulados = []
    for i, row in grupo.iterrows():
        if row["Nivel Real"] > 0:
            acumulados.append(row["Acumulado Real"])  # usamos real mientras exista
            acumulado = row["Acumulado Real"]
        else:
            acumulado += row["Nivel Proyectado"]
            acumulados.append(acumulado)
    
    proyamb_dia.loc[grupo.index, "Acumulado Proyectado"] = acumulados

proyamb_dia.rename(columns={
    'Rubro Presupuesto': 'Rubro'}, inplace=True)

proyamb_dia['Acumulado Proyectado'] = proyamb_dia['Acumulado Proyectado'].fillna(0).astype(int)

columnas_finales = ['Fecha', 'Rubro', 'Subrubro', 'Nivel Real', 'Nivel Proyectado', 'Acumulado Real', 'Acumulado Proyectado','Nivel Esperado']
proyamb_dia = proyamb_dia[columnas_finales]

# Ajustar acumulados según la fecha respecto a `hoy`

# Acumulado Real: mantener solo para fechas pasadas/igual a hoy, sino 0
if 'Acumulado Real' in proyamb_dia.columns:
    proyamb_dia['Acumulado Real'] = proyamb_dia['Acumulado Real'].where(proyamb_dia['Fecha'] < hoy, 0)

# Acumulado Proyectado: mantener solo para fechas futuras (> hoy), sino 0
if 'Acumulado Proyectado' in proyamb_dia.columns:
    proyamb_dia['Acumulado Proyectado'] = proyamb_dia['Acumulado Proyectado'].where(proyamb_dia['Fecha'] >= hoy, 0)

#### Proyección del Ejercicio
- __Intervalo de tiempo__: meses del Ejercicio 2025-2026   
- __Apertura__: Rubro - Subrubro - Zona - Subzona  
- __Métricas__: Nivel Real - Nivel Proyectado - Nivel Esperado

In [10]:
# Calculando proyección del ejercicio - ambulatorio
prestaciones = ambulatorio_wtf.groupby(['Rubro Presupuesto', 'Subrubro Presupuesto', 'Zona', 'Subzona', 'Periodo'])['Prestaciones'].sum().reset_index() # Agrupar y sumar Prestaciones
prestaciones.rename(columns={
    'Rubro Presupuesto': 'Rubro',
    'Subrubro Presupuesto': 'Subrubro'
}, inplace=True) # Renombrar columnas

# === Niveles Esperados ===
proyejamb = auxiliares['amb'].merge(prestaciones, on=['Rubro', 'Subrubro', 'Zona', 'Subzona','Periodo'], how='left') 
proyejamb = (
    proyejamb
    .fillna({'Nivel Esperado': 0, 'Prestaciones': 0})
    .astype({'Prestaciones': int})
)

# === INCIDENCIA AMBULATORIO ===
# Para el cálculo de la incidencia se tienen en cuenta todas las prestaciones hasta el periodo anterior al actual.
inc_amb = ambulatorio_wtf[ambulatorio_wtf['Periodo']<periodo_actual].groupby(['Rubro Presupuesto','Subrubro Presupuesto','Zona', 'Subzona'])['Prestaciones'].sum().reset_index()
# Total de prestaciones por rubro
inc_amb["Total Subrubro"] = inc_amb.groupby("Subrubro Presupuesto")["Prestaciones"].transform("sum")
# Incidencia respecto al total de su rubro
inc_amb["Incidencia"] = (inc_amb["Prestaciones"] / inc_amb["Total Subrubro"])

ultimos_dict = proyamb_dia.groupby("Subrubro")["Acumulado Proyectado"].last().to_dict()

proyejamb['Aut. Proyectadas'] = proyejamb['Subrubro'].map(ultimos_dict)

# Merge entre proyejamb e inc_amb para traer incidencia por Zona/Subzona
proyejamb = proyejamb.merge(
    inc_amb[['Rubro Presupuesto','Subrubro Presupuesto','Zona','Subzona','Incidencia','Total Subrubro','Prestaciones']],
    left_on=['Rubro','Subrubro','Zona','Subzona'],
    right_on=['Rubro Presupuesto','Subrubro Presupuesto','Zona','Subzona'],
    how='left',
    suffixes=('','_inc')
)

# Normalizar resultados: rellenar NaN y limpiar columnas auxiliares
proyejamb['Incidencia'] = proyejamb['Incidencia'].fillna(0)
proyejamb['Total Subrubro'] = proyejamb['Total Subrubro'].fillna(0)
# Si se generaron columnas de prestaciones duplicadas, mantener la original de proyejamb
if 'Prestaciones_inc' in proyejamb.columns and 'Prestaciones' in proyejamb.columns:
    proyejamb = proyejamb.drop(columns=['Prestaciones_inc'])
# Eliminar columnas de mapeo usadas sólo para el merge
proyejamb = proyejamb.drop(columns=['Rubro Presupuesto','Subrubro Presupuesto'], errors='ignore')

# Distribuir 'Aut. Proyectadas' por zona/subzona usando la incidencia
if hoy.day > 10: #CHEQUEAR DESPUES DE QUE DIA QUIERO MOSTRAR LAS PROYECTADAS
    if 'Aut. Proyectadas' in proyejamb.columns:
    # Asegurar valores numéricos y no nulos
        proyejamb['Aut. Proyectadas'] = pd.to_numeric(proyejamb['Aut. Proyectadas'], errors='coerce').fillna(0)
        proyejamb['Incidencia'] = pd.to_numeric(proyejamb['Incidencia'], errors='coerce').fillna(0)

    # Aplicar distribución sólo para el periodo actual
        mask_actual = (proyejamb['Periodo'] == periodo_actual)
        proyejamb.loc[mask_actual, 'Aut. Proyectadas'] = (
        (proyejamb.loc[mask_actual, 'Aut. Proyectadas'] * proyejamb.loc[mask_actual, 'Incidencia'])
            .round()
            .astype(int)
    )
    # Mantener valores originales (enteros) para los demás periodos
        proyejamb.loc[~mask_actual, 'Aut. Proyectadas'] = proyejamb['Prestaciones']   

else:
        proyejamb['Aut. Proyectadas'] = 0
        mask_actual = (proyejamb['Periodo'] == periodo_actual)
        proyejamb.loc[~mask_actual, 'Aut. Proyectadas'] = proyejamb['Prestaciones']

    

proyejamb = proyejamb[['Rubro','Subrubro','Zona','Subzona','Periodo','Nivel Esperado','Prestaciones','Aut. Proyectadas']]

# Calculo del desvio proyectado
aux_desvio_amb = proyejamb.groupby(['Periodo','Rubro','Subrubro','Zona','Subzona'])[['Nivel Esperado', 'Aut. Proyectadas','Prestaciones']].sum().reset_index()
aux_desvio_amb['Desvio Proyectado'] = ((aux_desvio_amb['Aut. Proyectadas']/aux_desvio_amb['Nivel Esperado'])-1).round(6)

a = pd.merge(left=aux_desvio_amb, right=auxiliares['val_amb'],how='left', on=['Periodo','Rubro','Subrubro','Zona','Subzona'])
a['Faltante'] = a['Aut. Proyectadas'] - a['Prestaciones']
a['Dif. valorizada Periodo Prestación'] = a['Conversor']*a['VU']*(a['Aut. Proyectadas']-a['Nivel Esperado'])
a['Dif. valorizada'] =(a['M2']*a['Dif. valorizada Periodo Prestación']).fillna(0)


cols_in = ['Rubro','Subrubro','Zona','Subzona','Periodo','Nivel Esperado','Prestaciones','Aut. Proyectadas','Faltante','Dif. valorizada']
a = a[cols_in]

<h3>FARMACIA & VACUNAS</h3>

<h3> Proyección diaria</h3>
<ul>
<li><strong>Intervalo de tiempo</strong>: días del corriente mes.</li>   
<li><strong>Apertura</strong>: Rubro(Farmacia)</li>  
<li><strong>Métricas</strong>: Nivel Real - Nivel Proyectado - Nivel Esperado - Acumulado Real - Acumulado Proyectado</li> 
</ul>
<br>
Para calcular los Niveles Proyectados se toman los últimos 28 días sin contar el actual.

In [12]:
# Calculando proyección diaria de farmacia (únicamente)
farmvac_wt_proy = farmvac_wt.copy() # Copia de farmvac_wt
farmvac_wt_proy = farmvac_wt_proy[farmvac_wt_proy['Rubro'] == 'Farmacia'] # Filtrar por 'Farmacia'
farmvac_proy = farmvac_wt_proy.groupby(['Periodo', 'Fecha'], as_index=False)['Cantidad'].sum() # Agrupar por 'Periodo' y 'Fecha'
farmvac_proy = farmvac_proy[farmvac_proy['Fecha'] >= dias_proy] # Filtrar por fecha mayor o igual a 28 días atrás

# === Agregar columna día de la semana ===
# Diccionario de traducción de días
dias_espanol = {
    'Monday': 'lunes',
    'Tuesday': 'martes',
    'Wednesday': 'miércoles',
    'Thursday': 'jueves',
    'Friday': 'viernes',
    'Saturday': 'sábado',
    'Sunday': 'domingo'
}

# Agregar columna 'Dia' con el nombre del día de la semana
farmvac_proy['Dia'] = farmvac_proy['Fecha'].dt.day_name().map(dias_espanol)

# Reemplazar por "feriado" si la fecha está en el calendario
#farmvac_proy['Dia'] = farmvac_proy.apply( lambda row: 'feriado' if row['Fecha'].date() in feriados else row['Dia'], axis=1 )
farmvac_proy.loc[farmvac_proy['Fecha'].isin(feriados), 'Dia'] = 'feriado'

# === Calcular promedio por día de la semana ===
promedios = farmvac_proy.groupby('Dia')['Cantidad'].mean()
dias_orden = ['lunes', 'martes', 'miércoles', 'jueves', 'viernes', 'sábado', 'domingo','feriado']
promedios = promedios.reindex(dias_orden).round().astype('Int64')

# Crear rango de fechas para la proyección
ultima_fecha_farmvac = farmvac_proy['Fecha'].max()
inicio_mes = ultima_fecha_farmvac.replace(day=1)
fin_mes = ultima_fecha_farmvac + pd.offsets.MonthEnd(0)  # Último día del mes

# Asegurar que fin_mes incluya al menos hoy
if fin_mes < hoy:
    fin_mes = hoy + pd.offsets.MonthEnd(0)

# Crear tabla con todos los días del mes
proyeccion = pd.DataFrame({
    'Fecha': pd.date_range(start=inicio_mes, end=fin_mes)
})

proyeccion['Dia'] = proyeccion['Fecha'].dt.day_name().map(dias_espanol)

# === Columna 'Niveles Reales' ===
# Unir proyección con los datos reales de farmvac
farmvac_nr = farmvac_wt_proy.groupby('Fecha')['Cantidad'].sum().reset_index()  # Agrupar por fecha y sumar cantidades

proyeccion = proyeccion.merge(
    farmvac_nr[['Fecha', 'Cantidad']],
    on='Fecha',
    how='left'
).rename(columns={'Cantidad': 'Niveles Reales'})

# Si hay días sin coincidencia, rellena con 0
proyeccion['Niveles Reales'] = proyeccion['Niveles Reales'].fillna(0).astype(int)

# === Columna 'Niveles Proyectado' ===
proyeccion['Niveles Proyectado'] = proyeccion['Dia'].map(promedios) # Mapea los promedios según el día de la semana (los feriados quedan excluidos)
mask_feriado_puro = proyeccion['Fecha'].isin(feriado_puro)
proyeccion['Niveles Proyectado'] = np.where(mask_feriado_puro, (proyeccion['Niveles Proyectado']*0.10).round(), proyeccion['Niveles Proyectado'])
mask_feriado_nl = proyeccion['Fecha'].isin(no_laborables)
proyeccion['Niveles Proyectado'] = np.where(mask_feriado_nl, (proyeccion['Niveles Proyectado']*0.50).round(), proyeccion['Niveles Proyectado'])
mask_fiestas = proyeccion['Fecha'].isin(fiestas)
proyeccion["Niveles Proyectado"] = np.where(mask_fiestas, (proyeccion["Niveles Proyectado"]*0.25).round(), proyeccion["Niveles Proyectado"])

# === Columna 'Acumulado Real' ===
proyeccion['Acumulado Real'] = proyeccion['Niveles Reales'].cumsum()
proyeccion.loc[proyeccion['Fecha'] >= hoy, 'Acumulado Real'] = 0

# === Columna 'Acumulado Proyectado' ===
proyeccion['Acumulado Proyectado'] = 0 

# Encontrar el último acumulado real antes de que empiece el proyectado
real_data = proyeccion.loc[proyeccion['Fecha'] < hoy, 'Acumulado Real']
ultimo_real = real_data.iloc[-1] if len(real_data) > 0 else 0

# Encontrar el índice donde empieza el proyectado con manejo de error
hoy_rows = proyeccion[proyeccion['Fecha'] == hoy]
if len(hoy_rows) > 0:
    idx_inicio_proj = hoy_rows.index[0]
    # Sumar el último acumulado real al primer proyectado
    proyeccion.loc[idx_inicio_proj, 'Acumulado Proyectado'] += ultimo_real + proyeccion.loc[idx_inicio_proj, 'Niveles Proyectado']
    
    # Calcular acumulado hacia abajo sumando cada Niveles Proyectado
    for i in range(idx_inicio_proj + 1, len(proyeccion)):
        proyeccion.loc[i, 'Acumulado Proyectado'] = proyeccion.loc[i-1, 'Acumulado Proyectado'] + proyeccion.loc[i, 'Niveles Proyectado']
else:
    # Si no existe hoy en el rango (no debería ocurrir con el ajuste de fin_mes), copiar el último real
    proyeccion['Acumulado Proyectado'] = proyeccion['Acumulado Real'].copy()

# Cargar niveles esperados desde 'auxiliar' en la columna 'Nivel Esperado'
niv_esp = pd.read_excel('auxiliar.xlsx', sheet_name='farmvac')
sumnivelesp = niv_esp[niv_esp['Periodo'] == periodo_actual].groupby('Periodo')['Nivel Esperado'].sum().values[0]
proyeccion['Nivel Esperado'] = sumnivelesp.astype(int)

proyeccion['Rubro'] = 'Farmacia'
columnas_finales_farmvac = ['Fecha', 'Rubro', 'Niveles Reales', 'Niveles Proyectado', 'Acumulado Real', 'Acumulado Proyectado','Nivel Esperado']
proyeccion = proyeccion[columnas_finales_farmvac]


<h3>Proyección del Ejercicio</h3>
<ul>
<li><strong>Intervalo de tiempo</strong>: meses del Ejercicio 2025-2026</li>
<li><strong>Apertura</strong>: Rubro(Farmacia) - Zona - Subzona</li>
<li><strong>Métricas</strong>: Nivel Real - Nivel Proyectado - Nivel Esperado</li>
</ul>

In [13]:
# Calculando proyección del ejercicio - farmacia y vacunas
# === INCIDENCIA POR ZONA Y SUBZONA ===
# Agrupar por Zona y Subzona, sumar Cantidad y calcular porcentaje del total de Autorizaciones de Farmacia (Únicamente)
incidencia = farmvac_wt_proy.groupby(['Zona', 'Subzona'])['Cantidad'].sum().reset_index()
total = incidencia["Cantidad"].sum()
incidencia['Porcentaje del total'] = round((incidencia["Cantidad"]/total), 4)

ultimo_valor = proyeccion['Acumulado Proyectado'].iat[-1] # Último valor de la columna Acumulado Proyectado
incidencia['Aut. Proyectadas'] = (incidencia['Porcentaje del total']*ultimo_valor) # Proyección de Autorizaciones
incidencia['Aut. Proyectadas'] = incidencia['Aut. Proyectadas'].fillna(0).astype(int)
incidencia['Periodo'] = periodo_actual

# --- Agrupar por dimensiones clave y sumar cantidad ---
farmvac_agrupado = (
    farmvac_wt
    .groupby(['Rubro', 'Zona', 'Subzona', 'Periodo'], as_index=False)
    .agg({'Cantidad': 'sum'})
)

# --- Cargar auxiliar de nivel esperado ---
aux_farmvac = pd.read_excel('auxiliar.xlsx', sheet_name='farmvac')

# Convertir 'Nivel Esperado' a entero de forma segura
aux_farmvac['Nivel Esperado'] = pd.to_numeric(
    aux_farmvac['Nivel Esperado'], errors='coerce'
).fillna(0).astype(int)

# --- Unir niveles esperados con la tabla agrupada ---
farmvac_sheet = aux_farmvac.merge(
    farmvac_agrupado,
    on=['Rubro', 'Zona', 'Subzona', 'Periodo'],
    how='left'
)

# Hacer merge solo con las columnas necesarias
farmvac_sheet = farmvac_sheet.merge(
    incidencia[['Zona', 'Subzona','Periodo','Aut. Proyectadas']],
    on=['Zona', 'Subzona','Periodo'],
    how='left'
)

# Valores NaN en 'Cantidad' y 'Aut. Proyectadas' se llenan con 0 y se convierten a entero
farmvac_sheet['Cantidad'] = farmvac_sheet['Cantidad'].fillna(0).astype(int)
farmvac_sheet['Aut. Proyectadas'] = farmvac_sheet['Aut. Proyectadas'].fillna(0).astype(int)

# Aplicar condición para que si el periodo es distinto del actual y el dia menor que 11 se muestren las proyectadas
# Condición 1: Rubro == Farmacia
cond_rubro = farmvac_sheet["Rubro"] == "Farmacia"
# Condición 2: periodo actual
cond_periodo = farmvac_sheet["Periodo"] == periodo_actual

if hoy.day > 10:
    farmvac_sheet['Aut. Proyectadas'] = farmvac_sheet['Aut. Proyectadas'].where(cond_periodo,farmvac_sheet['Cantidad'])
    farmvac_sheet['Aut. Proyectadas'] = farmvac_sheet['Aut. Proyectadas'].where(cond_rubro,0)
else:
    farmvac_sheet['Aut. Proyectadas'] = 0
    farmvac_sheet['Aut. Proyectadas'] = farmvac_sheet['Aut. Proyectadas'].where(cond_periodo,farmvac_sheet['Cantidad'])
    farmvac_sheet['Aut. Proyectadas'] = farmvac_sheet['Aut. Proyectadas'].where(cond_rubro,0)

In [14]:
# Farmacia y Vacunas - Importe
fv_imp = farmvac.groupby(['Rubro','Zona','Subzona','Periodo'])['Importe'].sum().reset_index()
fvneimp = aux_farmvac[['Rubro','Zona','Subzona','Periodo','Nivel Esperado $']]
farmvac_imp = pd.merge(fv_imp, fvneimp, how='right',on=['Rubro','Zona','Subzona','Periodo'])
farmvac_imp = farmvac_imp.fillna(0)

In [15]:
# Proyección Importe
# --------------------------------------------------------
# 1. CÁLCULO DE VARIABLES DIARIAS Y PROYECCIONES
# --------------------------------------------------------

# Cantidades faltantes para llegar al acumulado proyectado
q_faltantes = proyeccion['Acumulado Proyectado'].iloc[-1] - proyeccion.loc[proyeccion['Fecha'] == ayer, 'Acumulado Real'].iloc[0]

# Nivel esperado acumulado para el periodo actual
ne_farm_imp = farmvac_imp[(farmvac_imp['Periodo'] == periodo_actual) & (farmvac_imp['Rubro'] == 'Farmacia')]
total_nivel_esperado = ne_farm_imp['Nivel Esperado $'].sum().round(2)

# Proyección del importe faltante para llegar al acumulado proyectado
farm_proy_imp = farmvac[(farmvac['Periodo'] == periodo_actual) & (farmvac['Rubro'] == 'Farmacia')]
farm_proy_imp = farm_proy_imp.groupby(['Fecha'])[['Cantidad', 'Importe']].sum().reset_index()
farm_proy_imp = farm_proy_imp[farm_proy_imp['Fecha'] < hoy]

# Cálculos de proyecciones
promedio_dia = (farm_proy_imp['Importe'] / farm_proy_imp['Cantidad']).mean()
total_importe_real = farm_proy_imp['Importe'].sum()
proyeccion_importe = ((promedio_dia * q_faltantes) + total_importe_real).round(2)


# --------------------------------------------------------
# 2. SISTEMA DE HISTORIAL Y GUARDADO
# --------------------------------------------------------

archivo_datos = 'data_temp/historial_seg_farm_proy.csv'
fecha_hoy = pd.Timestamp.now().normalize()

# A. Intentar cargar los datos históricos
if os.path.exists(archivo_datos):
    seg_farm_proy_imp = pd.read_csv(archivo_datos, parse_dates=['Fecha'])
else:
    seg_farm_proy_imp = pd.DataFrame(columns=['Fecha', 'Real', 'Importe Proyectado', 'Nivel Esperado'])

# B. Crear la fila con los datos calculados hoy
nueva_data = pd.DataFrame({
    'Fecha': [fecha_hoy],
    'Real': [total_importe_real],
    'Importe Proyectado': [proyeccion_importe],
    'Nivel Esperado': [total_nivel_esperado]
})

# C. Actualizar si ya corriste el código hoy, o agregar si es la primera vez
if not seg_farm_proy_imp.empty and (seg_farm_proy_imp['Fecha'] == fecha_hoy).any():
    idx = seg_farm_proy_imp.index[seg_farm_proy_imp['Fecha'] == fecha_hoy][0]
    seg_farm_proy_imp.loc[idx, ['Real', 'Importe Proyectado', 'Nivel Esperado']] = \
        [nueva_data['Real'].values[0], nueva_data['Importe Proyectado'].values[0], nueva_data['Nivel Esperado'].values[0]]
    print("🔄 Datos de hoy actualizados correctamente.")
else:
    seg_farm_proy_imp = pd.concat([seg_farm_proy_imp, nueva_data], ignore_index=True)
    print("✅ Nueva fecha registrada en el historial.")

# D. Guardar el archivo en el disco
seg_farm_proy_imp.to_csv(archivo_datos, index=False)

✅ Nueva fecha registrada en el historial.


In [16]:
# Agregamos la proyección del importe al DataFrame de farmvac_imp
# Extraemos el último valor proyectado (como vimos en el paso anterior)
ultimo_proyectado = seg_farm_proy_imp['Importe Proyectado'].iloc[-1]

# Usamos .loc[condicion_filas, nombre_columna] = valor
farmvac_imp.loc[(farmvac_imp['Periodo'] == periodo_actual) & (farmvac_imp['Rubro'] == 'Farmacia'), 'Proyección Importe'] = ultimo_proyectado

### PROVISIÓN

In [17]:
# Calculando provisión
# Auxiliar con los niveles esperados
aux_provision = pd.read_excel('auxiliar.xlsx', sheet_name='provisión')
acreedor_alimentos = ['785833','719848','108077','616554','203401','192794','169179','47359','644229','20005','781897','115355','68378','46433']
provision['Provision'] = provision['Provision'].str.strip() # Eliminar espacios en blanco al inicio y al final

# 1. Definimos las condiciones en orden de prioridad
condiciones = [
    provision['Provision'] == 'MEDICAMENTOS ESPECIALES',  # Caso A
    provision['Acreedor_id'].isin(acreedor_alimentos)  # Caso B
]

# 2. Definimos qué valor poner para cada caso
elecciones = [
    'DROGUERIA',  # Resultado para Caso A
    'ALIMENTOS'   # Resultado para Caso B
]

# 3. Ejecutamos (default es lo que pone si no cumple ninguna anterior)
provision['NR'] = np.select(condiciones, elecciones, default='OTRAS')

provision = provision.groupby(['Periodo','NR','Origen'])[['Importe','Cantidad']].sum().reset_index()

# Asegurar que las columnas usadas para merge tengan tipos compatibles
aux_provision['Provision AC'] = aux_provision['Provision AC'].astype(str).str.strip()
aux_provision['Origen'] = aux_provision['Origen'].astype(str).str.strip()
provision['NR'] = provision['NR'].astype(str).str.strip()
provision['Origen'] = provision['Origen'].astype(str).str.strip()


# Unión entre la tabla auxiliar y la tabla de provisión
# Nota: alinear el orden de left_on y right_on para evitar merges entre tipos incompatibles
provision_sheet = pd.merge(
    aux_provision,
    provision,
    left_on=['Provision AC', 'Origen', 'Periodo'],
    right_on=['NR', 'Origen', 'Periodo'],
    how='left'
)

provision_sheet = provision_sheet.rename(columns={
    'Cantidad': 'Autorizaciones QTY',
    'Importe': 'Autorizaciones $'  
})

provision_sheet['Rubro'] = 'Provisión'
orden = ['Rubro','Provision AC','Origen','Periodo','$ Nivel Esperado', 'Autorizaciones $',  'Autorizaciones QTY']
provision_sheet = provision_sheet[orden] # Reordenar columnas
provision_sheet = provision_sheet.fillna(0)

In [18]:
# Calculando provisión diaria
from mstrio.project_objects import Report
from utils.login import conn

prov_diario = Report(id='88653689CB40D7DDC63AE1983E1206BD', connection=conn).to_dataframe()
prov_d = prov_diario.groupby(['Fecha Proceso Autorización','Origen Autorización'])[['Importe Comprobante Prestacion','Cantidad Prestaciones Aceptadas']].sum().reset_index()
prov_d.rename(columns={
    'Fecha Proceso Autorización': 'Fecha',
    'Importe Comprobante Prestacion': 'Autorizaciones $',
    'Cantidad Prestaciones Aceptadas': 'Autorizaciones QTY'
}, inplace=True)

prov_d['Fecha'] = pd.to_datetime(prov_d['Fecha'])
prov_d = prov_d.sort_values('Fecha').reset_index(drop=True) # Asegurar que la columna de fecha esté en formato datetime y ordenada
prov_d['Origen Autorización'] = prov_d['Origen Autorización'].str.strip() # Eliminar espacios en blanco

Report object named: '05 - Autorizaciones MPROV_diario' with ID: '88653689CB40D7DDC63AE1983E1206BD'


In [19]:
# Identificar el mes y el año de tu dataset (tomamos el primer registro como referencia)
anio = prov_d['Fecha'].dt.year.iloc[0]
mes = prov_d['Fecha'].dt.month.iloc[0]

# Crear un rango con TODOS los días de ese mes (del día 1 al último)
inicio_mes = pd.Timestamp(year=anio, month=mes, day=1)
fin_mes = inicio_mes + pd.offsets.MonthEnd(1)
rango_completo_mes = pd.date_range(start=inicio_mes, end=fin_mes)

# Identificar cuáles de esos días son hábiles (Lunes a Viernes, sin feriados)
# Convertir explícitamente los feriados a DatetimeIndex para evitar FutureWarning
feriados_ar = pd.to_datetime(list(holidays.Argentina(years=anio).keys()))
es_dia_habil = (rango_completo_mes.dayofweek < 5) & (~rango_completo_mes.isin(feriados_ar))

# 1. Crear un DataFrame "Calendario" con TODOS los días del mes
df_calendario = pd.DataFrame({
    'Fecha': rango_completo_mes
})

# 2. Re-evaluamos la condición de día hábil directamente sobre la columna Fecha
feriados_ar = pd.to_datetime(list(holidays.Argentina(years=anio).keys()))
es_dia_habil = (df_calendario['Fecha'].dt.dayofweek < 5) & (~df_calendario['Fecha'].isin(feriados_ar))

# 3. Asignar el número de día hábil
# cumsum() va sumando 1 cada vez que encuentra un True (día hábil).
df_calendario['Dia_Habil_Mes'] = es_dia_habil.cumsum()

# Opcional: Si quieres que los fines de semana/feriados muestren NaN (como en tu ejemplo)
df_calendario.loc[~es_dia_habil, 'Dia_Habil_Mes'] = np.nan

# 4. Cruzar el calendario completo con tu dataset
# Ponemos df_calendario a la izquierda para que NINGÚN día del mes se pierda
prov_d = pd.merge(df_calendario, prov_d, on='Fecha', how='left')

# 5. Cálculos finales
# Buscamos el total en el calendario original (que es el "perfecto")
total_dias_habiles = df_calendario['Dia_Habil_Mes'].max() 

# Buscamos el último día hábil con datos (ignorando los nulos que acabamos de generar en los días sin datos)
dia_habil_actual = prov_d.dropna(subset=['Autorizaciones $'])['Dia_Habil_Mes'].max()

# Si dia_habil_actual es NaN (porque quizás no hay datos aún), lo manejamos poniendo un 0
if pd.isna(dia_habil_actual):
    dia_habil_actual = 0

dias_restantes = total_dias_habiles - dia_habil_actual

print(f"Total de días hábiles: {total_dias_habiles}")
print(f"Día hábil actual: {dia_habil_actual}")
print(f"Días para terminar el mes: {dias_restantes}")

Total de días hábiles: 21.0
Día hábil actual: 13.0
Días para terminar el mes: 8.0


Proyección Drogueria

In [20]:
# AMBULATORIO
prvd = prov_d[prov_d['Origen Autorización'] == 'Ambulatorio'].copy().reset_index(drop=True)

p_drog = (prvd['Autorizaciones $'].sum() * dias_restantes)/dia_habil_actual if dia_habil_actual > 0 else 0
proy_drog = prvd['Autorizaciones $'].sum() + p_drog

p_drog_dia = p_drog / dias_restantes if dias_restantes > 0 else 0

# Ver el resultado
print(f"Días faltantes: {dias_restantes}")
print(f"Proyección de Droguería para Ambulatorio: {proy_drog}")
print(f"Proyección diaria de Droguería para Ambulatorio: {p_drog_dia}")

Días faltantes: 8.0
Proyección de Droguería para Ambulatorio: 14488924858.153847
Proyección diaria de Droguería para Ambulatorio: 689948802.7692307


In [21]:
# INTERNACION
prvd_int = prov_d[prov_d['Origen Autorización'] == 'Internación'].copy().reset_index(drop=True)

p_drog_int = (prvd_int['Autorizaciones $'].sum() * dias_restantes)/dia_habil_actual if dia_habil_actual > 0 else 0
proy_drog_int = prvd_int['Autorizaciones $'].sum() + p_drog_int

p_drog_int_dia = p_drog_int / dias_restantes if dias_restantes > 0 else 0

# Ver el resultado
print(f"Días faltantes: {dias_restantes}")
print(f"Proyección de Droguería para Internación: {proy_drog_int}")
print(f"Proyección diaria de Droguería para Internación: {p_drog_int_dia}")

Días faltantes: 8.0
Proyección de Droguería para Internación: 239383760.53846154
Proyección diaria de Droguería para Internación: 11399226.692307692


In [22]:
# Creamos la columna 'Proyección' vacía en todo el dataframe
prov_d['Proyección'] = np.nan

# Guardamos TODAS las fechas que tienen el Origen vacío (NaN) (tanto pasadas como futuras)
dias_vacios = prov_d[prov_d['Origen Autorización'].isna()][['Fecha', 'Dia_Habil_Mes']].drop_duplicates()

# Eliminamos esas filas incompletas del DataFrame original
prov_d = prov_d.dropna(subset=['Origen Autorización'])

# Construimos las filas nuevas correctamente desdobladas (Ambulatorio e Internación)
nuevas_filas = []
for _, row in dias_vacios.iterrows():
    # Evaluamos si la fecha es de hoy en adelante para aplicar la proyección
    # Si es pasada (ej. un fin de semana viejo), la proyección será 0
    proy_amb = p_drog_dia if row['Fecha'] >= hoy else 0
    proy_int = p_drog_int_dia if row['Fecha'] >= hoy else 0
    
    # Fila para Ambulatorio
    nuevas_filas.append({
        'Fecha': row['Fecha'], 
        'Dia_Habil_Mes': row['Dia_Habil_Mes'], 
        'Origen Autorización': 'Ambulatorio',
        'Proyección': proy_amb
    })
    # Fila para Internación
    nuevas_filas.append({
        'Fecha': row['Fecha'], 
        'Dia_Habil_Mes': row['Dia_Habil_Mes'], 
        'Origen Autorización': 'Internación',
        'Proyección': proy_int
    })

# Agregamos las filas nuevas al DataFrame
prov_d = pd.concat([prov_d, pd.DataFrame(nuevas_filas)], ignore_index=True)

# Actualizamos los días que SÍ tenían datos cargados pero son >= hoy (por si los hay)
prov_d.loc[(prov_d['Origen Autorización'] == 'Ambulatorio') & (prov_d['Fecha'] >= hoy), 'Proyección'] = p_drog_dia
prov_d.loc[(prov_d['Origen Autorización'] == 'Internación') & (prov_d['Fecha'] >= hoy), 'Proyección'] = p_drog_int_dia

# Rellenar con 0 las filas que no son futuras o que ya tenían datos
prov_d = prov_d.fillna(0)

# Finalmente, ordenamos por fecha y origen, y eliminamos la columna de día hábil
prov_d = prov_d.drop(columns=['Dia_Habil_Mes']).sort_values(['Fecha', 'Origen Autorización']).reset_index(drop=True)


### DISCAPACIDAD

In [23]:
# Calculando cantidades por Categoria, Zona, Subzona y Periodo.
# Unión entre la tabla auxiliar y la tabla de discapacidad, se reemplazan los rubros por los nombres correspondientes
aux_discapacidad = pd.read_excel('auxiliar.xlsx', sheet_name='discapacidad')
aux_discapacidad = discapacidad.merge(aux_discapacidad, on='Prestacion_id', how='left')

# Agrupar por Rubro, Categoria, Zona, Subzona, Periodo y sumar Cantidad
discapacidad_grouped = aux_discapacidad.groupby(
    ['Categoria', 'Zona', 'Subzona', 'Periodo'], as_index=False
)['Cantidad'].sum()

discapacidad_grouped.insert(0, 'Rubro', 'Discapacidad')

# Calcular periodo_actual
fecha_actual = datetime.today()
año = fecha_actual.year
mes = fecha_actual.month
periodo_anterior = periodo_actual - 1 if mes > 1 else (año - 1) * 100 + 12

# Filtrar la tabla para excluir los datos del periodo actual
df_sin_actual = discapacidad_grouped[discapacidad_grouped['Periodo'] != periodo_actual]

In [24]:
# Calculando promedio de los últimos 6 periodos con datos para cada combinación de Categoria, Zona y Subzona
# Para cada combinación de Categoria, Zona y Subzona, tomar los últimos 6 periodos con datos y calcular el promedio

def promedio_ultimos_6_periodos(df):
    df = df.copy()
    # Asegurar tipos correctos
    df['Periodo'] = pd.to_numeric(df['Periodo'], errors='coerce')
    df['Cantidad'] = pd.to_numeric(df['Cantidad'], errors='coerce')

    # Ordenar por periodo descendente
    df_sorted = df.sort_values('Periodo', ascending=False)

    resultados = []

    # Agrupar por Categoria, Zona y Subzona
    for (cat, zona, subzona), grupo in df_sorted.groupby(['Categoria', 'Zona', 'Subzona']):
        # Tomar los 6 periodos únicos más recientes
        ultimos_6_periodos = grupo['Periodo'].drop_duplicates().head(6)

        # Filtrar solo las filas correspondientes a esos periodos
        grupo_filtrado = grupo[grupo['Periodo'].isin(ultimos_6_periodos)]

        # Calcular promedio
        promedio = grupo_filtrado['Cantidad'].mean()

        # Convertir los periodos a string separados por coma
        periodos_str = ', '.join(str(p) for p in sorted(ultimos_6_periodos, reverse=True))

        # Guardar resultado
        resultados.append({
            'Categoria': cat,
            'Zona': zona,
            'Subzona': subzona,
            'Promedio_ultimos_6M': round(promedio) if pd.notnull(promedio) else pd.NA,
            'Periodos_utilizados': periodos_str
        })

    # Convertir a DataFrame y asegurar tipo Int64 para promedio
    resultado_df = pd.DataFrame(resultados)
    resultado_df['Promedio_ultimos_6M'] = resultado_df['Promedio_ultimos_6M'].astype('Int64')

    return resultado_df


promedio_ultimos_6 = promedio_ultimos_6_periodos(df_sin_actual)

# Agregar columna 'Periodo' con el valor del periodo anterior al actual
promedio_ultimos_6['Periodo'] = periodo_anterior

In [25]:
# Calculando promedio de los últimos 12 periodos con datos para cada combinación de Categoria, Zona y Subzona
def promedio_ultimos_12_periodos(df):
    df = df.copy()
    # Asegurar tipos correctos
    df['Periodo'] = pd.to_numeric(df['Periodo'], errors='coerce')
    df['Cantidad'] = pd.to_numeric(df['Cantidad'], errors='coerce')

    # Ordenar por periodo descendente
    df_sorted = df.sort_values('Periodo', ascending=False)

    resultados = []

    # Agrupar por Categoria, Zona y Subzona
    for (cat, zona, subzona), grupo in df_sorted.groupby(['Categoria', 'Zona', 'Subzona']):
        # Tomar los 12 periodos únicos más recientes
        ultimos_12_periodos = grupo['Periodo'].drop_duplicates().head(12)

        # Filtrar solo las filas correspondientes a esos periodos
        grupo_filtrado = grupo[grupo['Periodo'].isin(ultimos_12_periodos)]

        # Calcular promedio
        promedio = grupo_filtrado['Cantidad'].mean()

        # Convertir los periodos a string separados por coma (ordenados de más nuevo a más viejo)
        periodos_str = ', '.join(str(p) for p in sorted(ultimos_12_periodos, reverse=True))

        # Guardar resultado
        resultados.append({
            'Categoria': cat,
            'Zona': zona,
            'Subzona': subzona,
            'Promedio_ultimos_12M': round(promedio) if pd.notnull(promedio) else pd.NA,
            'Periodos_utilizados': periodos_str
        })

    # Convertir a DataFrame y asegurar tipo Int64 para promedio
    resultado_df = pd.DataFrame(resultados)
    resultado_df['Promedio_ultimos_12M'] = resultado_df['Promedio_ultimos_12M'].astype('Int64')

    return resultado_df

promedio_ultimos_12 = promedio_ultimos_12_periodos(df_sin_actual)

# Agregar columna 'Periodo' con el valor del periodo anterior al actual
promedio_ultimos_12['Periodo'] = periodo_anterior

In [26]:
# Uniendo los promedios calculados con la tabla principal
discapacidad_sheet = discapacidad_grouped.merge(
    promedio_ultimos_6[['Categoria', 'Zona', 'Subzona', 'Periodo', 'Promedio_ultimos_6M']],
    on=['Categoria', 'Zona', 'Subzona', 'Periodo'],
    how='left'
).merge(
    promedio_ultimos_12[['Categoria', 'Zona', 'Subzona', 'Periodo', 'Promedio_ultimos_12M']],
    on=['Categoria', 'Zona', 'Subzona', 'Periodo'],
    how='left'
)

# Mostrar solo los promedios y no los periodos utilizados
discapacidad_sheet = discapacidad_sheet[['Rubro', 'Categoria', 'Zona', 'Subzona', 'Periodo', 'Cantidad', 'Promedio_ultimos_6M', 'Promedio_ultimos_12M']]

### PROTESIS

In [27]:
# Calculando cantidades por Categoria, Zona, Subzona y Periodo.
protesis['Rubro General'] = "Prótesis"

protesis_agrupado = (protesis
    .groupby([
        'Rubro General',
        'Zona',
        'Subzona',
        'Periodo',
        'Origen'
    ], as_index=False)
    [['Cantidad', 'Importe']]
    .sum()
)

# Calcular periodo anterior
fecha_actual = datetime.today()
año = fecha_actual.year
mes = fecha_actual.month
periodo_anterior = periodo_actual - 1 if mes > 1 else (año - 1) * 100 + 12

# Filtrar la tabla para excluir los datos del periodo actual
protesis_sin_actual = protesis_agrupado[protesis_agrupado['Periodo'] != periodo_actual]

In [28]:
# Calculando promedio de los últimos 6 periodos con datos para cada combinación de Categoria, Zona y Subzona
# Para cada combinación de Categoria, Zona y Subzona, tomar los últimos 6 periodos con datos y calcular el promedio

def promedio_ultimos_6_periodos(df):
    df = df.copy()
    # Asegurar tipos correctos
    df['Periodo'] = pd.to_numeric(df['Periodo'], errors='coerce')
    df['Cantidad'] = pd.to_numeric(df['Cantidad'], errors='coerce')
    df['Importe'] = pd.to_numeric(df['Importe'], errors='coerce')

    # Ordenar por periodo descendente
    df_sorted = df.sort_values('Periodo', ascending=False)

    resultados = []

    # Agrupar por Categoria, Zona y Subzona
    for (zona, subzona, origen), grupo in df_sorted.groupby(['Zona', 'Subzona', 'Origen']):
        # Tomar los 6 periodos únicos más recientes
        ultimos_6_periodos = grupo['Periodo'].drop_duplicates().head(6)

        # Filtrar solo las filas correspondientes a esos periodos
        grupo_filtrado = grupo[grupo['Periodo'].isin(ultimos_6_periodos)]

        # Calcular promedio
        promedio = grupo_filtrado['Cantidad'].mean()
        promedio_importe = grupo_filtrado['Importe'].mean()

        # Convertir los periodos a string separados por coma
        periodos_str = ', '.join(str(p) for p in sorted(ultimos_6_periodos, reverse=True))

        # Guardar resultado
        resultados.append({
            'Zona': zona,
            'Subzona': subzona,
            'Origen': origen,
            'Promedio_ultimos_6M': round(promedio) if pd.notnull(promedio) else pd.NA,
            'Promedio_ultimos_6M_importe': round(promedio_importe) if pd.notnull(promedio_importe) else pd.NA,
            'Periodos_utilizados': periodos_str
        })

    # Convertir a DataFrame y asegurar tipo Int64 para promedio
    resultado_df = pd.DataFrame(resultados)
    resultado_df['Promedio_ultimos_6M'] = resultado_df['Promedio_ultimos_6M'].astype('Int64')
    resultado_df['Promedio_ultimos_6M_importe'] = resultado_df['Promedio_ultimos_6M_importe'].astype('Int64')

    return resultado_df


promedio_ultimos_6 = promedio_ultimos_6_periodos(protesis_sin_actual)
# Agregar columna 'Periodo' con el valor del periodo anterior al actual
promedio_ultimos_6['Periodo'] = periodo_anterior

In [29]:
# Calculando promedio de los últimos 12 periodos con datos para cada combinación de Categoria, Zona y Subzona
# Para cada combinación de Categoria, Zona y Subzona, tomar los últimos 6 periodos con datos y calcular el promedio

def promedio_ultimos_12_periodos(df):
    df = df.copy()
    # Asegurar tipos correctos
    df['Periodo'] = pd.to_numeric(df['Periodo'], errors='coerce')
    df['Cantidad'] = pd.to_numeric(df['Cantidad'], errors='coerce')
    df['Importe'] = pd.to_numeric(df['Importe'], errors='coerce')

    # Ordenar por periodo descendente
    df_sorted = df.sort_values('Periodo', ascending=False)

    resultados = []

    # Agrupar por Categoria, Zona y Subzona
    for (zona, subzona, origen), grupo in df_sorted.groupby(['Zona', 'Subzona', 'Origen']):
        # Tomar los 6 periodos únicos más recientes
        ultimos_12_periodos = grupo['Periodo'].drop_duplicates().head(12)

        # Filtrar solo las filas correspondientes a esos periodos
        grupo_filtrado = grupo[grupo['Periodo'].isin(ultimos_12_periodos)]

        # Calcular promedio
        promedio = grupo_filtrado['Cantidad'].mean()
        promedio_importe = grupo_filtrado['Importe'].mean()

        # Convertir los periodos a string separados por coma
        periodos_str = ', '.join(str(p) for p in sorted(ultimos_12_periodos, reverse=True))

        # Guardar resultado
        resultados.append({
            'Zona': zona,
            'Subzona': subzona,
            'Origen': origen,
            'Promedio_ultimos_12M': round(promedio) if pd.notnull(promedio) else pd.NA,
            'Promedio_ultimos_12M_importe': round(promedio_importe) if pd.notnull(promedio_importe) else pd.NA,
            'Periodos_utilizados': periodos_str
        })

    # Convertir a DataFrame y asegurar tipo Int64 para promedio
    resultado_df = pd.DataFrame(resultados)
    resultado_df['Promedio_ultimos_12M'] = resultado_df['Promedio_ultimos_12M'].astype('Int64')
    resultado_df['Promedio_ultimos_12M_importe'] = resultado_df['Promedio_ultimos_12M_importe'].astype('Int64')

    return resultado_df


promedio_ultimos_12 = promedio_ultimos_12_periodos(protesis_sin_actual)
# Agregar columna 'Periodo' con el valor del periodo anterior al actual
promedio_ultimos_12['Periodo'] = periodo_anterior

In [30]:
# Uniendo los promedios calculados con la tabla principal
protesis_sheet = protesis_agrupado.merge(
    promedio_ultimos_6[['Zona', 'Subzona', 'Origen', 'Periodo', 'Promedio_ultimos_6M', 'Promedio_ultimos_6M_importe']],
    on=['Zona', 'Subzona', 'Origen', 'Periodo'],
    how='left'
).merge(
    promedio_ultimos_12[['Zona', 'Subzona', 'Origen', 'Periodo', 'Promedio_ultimos_12M', 'Promedio_ultimos_12M_importe']],
    on=['Zona', 'Subzona', 'Origen', 'Periodo'],
    how='left'
)

ordenamiento_columnas = ['Rubro General', 'Zona', 'Subzona', 'Periodo', 'Origen', 'Cantidad',
       'Importe', 'Promedio_ultimos_6M', 'Promedio_ultimos_12M', 'Promedio_ultimos_6M_importe', 'Promedio_ultimos_12M_importe']

protesis_sheet = protesis_sheet[ordenamiento_columnas]

### Tablas para actualizar Google Sheets

In [31]:
# Creando archivo Excel con todas las hojas a exportar.
# concatenado para el seguimiento mes actual
proyeccion = proyeccion.rename(columns={
    'Niveles Reales':'Nivel Real',
    'Niveles Proyectado':'Nivel Proyectado'
})
segdiario = pd.concat([proyamb_dia,proyeccion])
segdiario['Fecha'] = segdiario['Fecha'].dt.date


with pd.ExcelWriter(f'files/gsheet_{dia_hoy}.xlsx', engine='openpyxl') as writer:
    a.to_excel(writer, sheet_name='Ambulatorio', index=False)
    farmvac_sheet.to_excel(writer, sheet_name='FarmVac', index=False)
    farmvac_imp.to_excel(writer, sheet_name='FarmVacImp',index=False)
    seg_farm_proy_imp.to_excel(writer, sheet_name='SegFarmProyImp', index=False)
    provision_sheet.to_excel(writer, sheet_name='Provision', index=False)
    prov_d.to_excel(writer, sheet_name='Provision Diario', index=False)
    discapacidad_sheet.to_excel(writer, sheet_name='Discapacidad', index=False)
    protesis_sheet.to_excel(writer, sheet_name='Protesis', index=False)
    proyamb_dia.to_excel(writer, sheet_name='ProyAmb', index=False)
    proyeccion.to_excel(writer, sheet_name='ProyFarm', index=False)
    segdiario.to_excel(writer, sheet_name='SegDiario', index=False)
    
print("Archivo Excel generado exitosamente. 🆗")
print(f'Archivo guardado en: files/gsheet_{dia_hoy}.xlsx 💾')

Archivo Excel generado exitosamente. 🆗
Archivo guardado en: files/gsheet_2026-06-19.xlsx 💾
